In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

text = """
To be or not to be, that is the question:
Whether tis nobler in the mind to suffer
The slings and arrows of outrageous fortune,
Or to take arms against a sea of troubles,
And by opposing end them.
To die, to sleep,
No more; and by a sleep to say we end
The heartache and the thousand natural shocks
That flesh is heir to, tis a consummation
Devoutly to be wished.
To die, to sleep;
To sleep, perchance to dream.
Ay, theres the rub.
"""

text = text.lower()

chars = sorted(list(set(text)))
vocab_size = len(chars)

char_to_int = {c: i for i, c in enumerate(chars)}
int_to_char = {i: c for i, c in enumerate(chars)}

encoded = [char_to_int[c] for c in text]

seq_len = 30

X = []
Y = []

for i in range(len(encoded) - seq_len):
    X.append(encoded[i:i + seq_len])
    Y.append(encoded[i + seq_len])

X = torch.tensor(X, dtype=torch.long)
Y = torch.tensor(Y, dtype=torch.long)

class LSTMModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, 64)
        self.lstm = nn.LSTM(64, 128, batch_first=True)
        self.fc = nn.Linear(128, vocab_size)

    def forward(self, x):
        x = self.embedding(x)
        x, _ = self.lstm(x)
        x = x[:, -1, :]
        x = self.fc(x)
        return x

model = LSTMModel(vocab_size)

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)

epochs = 50

for epoch in range(epochs):
    model.train()

    output = model(X)
    loss = loss_fn(output, Y)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch + 1}/{epochs} Loss: {loss.item():.4f}")

def generate(seed, length=200):
    model.eval()

    result = seed.lower()

    for _ in range(length):
        sequence = result[-seq_len:]

        if len(sequence) < seq_len:
            sequence = " " * (seq_len - len(sequence)) + sequence

        seq = torch.tensor(
            [char_to_int.get(c, 0) for c in sequence],
            dtype=torch.long
        ).unsqueeze(0)

        with torch.no_grad():
            prediction = model(seq)
            probabilities = F.softmax(prediction, dim=1)

        next_char = torch.multinomial(probabilities, 1).item()
        result += int_to_char[next_char]

    return result

print("\nGenerated Text:\n")
print(generate("to be or not to be", 200))

Epoch 5/50 Loss: 2.8046
Epoch 10/50 Loss: 2.4323
Epoch 15/50 Loss: 2.0679
Epoch 20/50 Loss: 1.7202
Epoch 25/50 Loss: 1.3523
Epoch 30/50 Loss: 1.0297
Epoch 35/50 Loss: 0.7160
Epoch 40/50 Loss: 0.4578
Epoch 45/50 Loss: 0.2688
Epoch 50/50 Loss: 0.1537

Generated Text:

to be or not to beem.
to die, to sheleshint uevot, ier to sleep,
no tlee, re are dr and by ois, theres the is aind the tion the mind to mfarea s of iend the thaks f of ageis to bef ther tis nobler is in the in to slee,
